In [1]:
 import numpy as np
from matplotlib import pyplot as plt

# In[]:
import opfunu  # 参考文档：https://github.com/thieu1995/opfunu
import mealpy
from APO import APO
from BSLO import BSLO
from MSGWO import MSGWO
from GWO import GWO
from HO import HO
from IVY import IVY
from BKA import BKA
from GA import GA
from NOA import NOA
from RBMO import RBMO
from SBOA import SBOA
# from mealpy.swarm_based import WOA, GWO
from mealpy import get_optimizer_by_name
from mealpy.evolutionary_based.GA import BaseGA
plt.rcParams['font.family'] = 'Times New Roman'

'''
适应度函数及维度dim的选择
cec函数名字格式：函数名+年份，比如要选择2022的F1函数，func_num = 'F1'+'2022'
cec2005：F1-F25, 可选 dim = 10, 30, 50
cec2008：F1-F7,  可选 2 <= dim <= 1000
cec2010：F1-F20, 可选 100 <= dim <= 1000
cec2013：F1-F28, 可选 dim = 2, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100
cec2014：F1-F30, 可选 dim = 10, 20, 30, 50, 100
cec2015：F1-F15, 可选 dim = 10, 30
cec2017：F1-F29, 可选 dim = 2, 10, 20, 30, 50, 100
cec2019：F1-F10, 可选 dim: F1=9,F2=16,F3=18,其他=10
cec2020：F1-F10, 可选 dim = 2, 5, 10, 15, 20, 30, 50, 100
cec2021：F1-F10, 可选 dim = 2, 10, 20
cec2022：F1-F12, 可选 dim = 2, 10, 20

'''
fun_name = 'F28'  # 按需修改
year = '2017'  # 按需修改
func_num = fun_name + year
dim = 30  # 维度，根据cec函数 选择对应维度
epoch = 100  # 最大迭代次数
pop_size = 30  # 种群数量
'''定义的 cec函数 '''


def cec_fun(x):
    funcs = opfunu.get_functions_by_classname(func_num)
    func = funcs[0](ndim=dim)
    F = func.evaluate(x)
    return F


''' fit_func->目标函数, lb->下限, ub->上限 '''
problem_dict = {
    "fit_func": cec_fun,
    "lb": opfunu.get_functions_by_classname(func_num)[0](ndim=dim).lb.tolist(),
    "ub": opfunu.get_functions_by_classname(func_num)[0](ndim=dim).ub.tolist(),
    "minmax": "min",
}

In [2]:
# import pandas as pd
# import os
# from openpyxl import load_workbook
# 
# def append_to_excel(filename, data_list):
#     # 将列表转换为DataFrame
#     df = pd.DataFrame([data_list])
# 
#     # 检查文件是否存在
#     if not os.path.isfile(filename):
#         # 如果文件不存在，则创建新文件并写入数据
#         df.to_excel(filename, index=False, header=False)
#     else:
#         # 如果文件存在，则追加数据
#         book = load_workbook(filename)
#         writer = pd.ExcelWriter(filename, engine='openpyxl')
#         writer.book = book
#         writer.sheets = {ws.title: ws for ws in book.worksheets}
# 
#         # 找到当前表中的最后一行
#         startrow = writer.sheets['Sheet1'].max_row
# 
#         df.to_excel(writer, startrow=startrow, index=False, header=False)
# 
#         writer.save()
# 
# # # 示例数据列表
# # data_list = [1, 2, 3, 4, 5]


In [3]:
import os
import pandas as pd
from openpyxl import load_workbook

def append_to_excel(filename, data_list):
    # 将列表转换为DataFrame
    df = pd.DataFrame([data_list])

    # 检查文件是否存在
    if not os.path.isfile(filename):
        # 如果文件不存在，创建新文件并写入数据，包含表头
        df.to_excel(filename, index=False, header=True)
    else:
        # 文件存在，尝试追加数据
        try:
            # 加载现有工作簿，并指定只读取数据，不读取样式等
            book = load_workbook(filename, data_only=True)

            # 确保Sheet1存在
            if 'Sheet1' not in book.sheetnames:
                book.create_sheet('Sheet1')

            # 找到Sheet1中的最后一行
            startrow = book['Sheet1'].max_row

            # 使用pandas的ExcelWriter以追加模式写入数据
            with pd.ExcelWriter(filename, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
                # 将数据追加到Sheet1
                df.to_excel(writer, sheet_name='Sheet1', startrow=startrow, index=False, header=False)

        except Exception as e:
            print(f"在追加数据时发生错误: {e}")

# # 示例数据列表
# data_list = [1, 2, 3, 4, 5]  # 注意这里应该是二维列表
# filename = "results.xlsx"
# # 追加数据到Excel文件
# append_to_excel(filename, data_list)
# 
# print(f"数据已成功写入 {filename}")

In [4]:
''' 调用优化算法 '''
''' 第二种方式，需：from mealpy import get_optimizer_by_name'''
woa_model = get_optimizer_by_name("OriginalWOA")(epoch, pop_size)
pso_model = get_optimizer_by_name("OriginalPSO")(epoch, pop_size)
ga_model = get_optimizer_by_name("BaseGA")(epoch, pop_size)

'''求解 cec函数 '''
H = []
F = []
f = []
for i in range(30):
    woa_best_x, woa_best_f = woa_model.solve(problem_dict)
    f.append(woa_best_f)
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    pso_best_x, pso_best_f = pso_model.solve(problem_dict)
    f.append(pso_best_f)
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve1, best_individual1 = GWO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve1[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve4, best_individual4 = HO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve4[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve5, best_individual5 = BSLO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve5[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve6, best_individual6 = RBMO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve6[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve7, best_individual7 = APO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve7[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve8, best_individual8 = IVY(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve8[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve9, best_individual9 = BKA(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve9[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve10, best_individual10 = GA(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve10[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    print("*****************************")
    fitness_curve, best_individual = MSGWO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
print(F)

2024/07/17 02:16:17 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: Solving single objective optimization problem.
2024/07/17 02:16:18 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 1, Current best: 190624727684486.3, Global best: 190624727684486.3, Runtime: 0.57890 seconds
2024/07/17 02:16:18 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 2, Current best: 164185472046893.22, Global best: 164185472046893.22, Runtime: 0.53003 seconds
2024/07/17 02:16:19 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 3, Current best: 68747796901005.766, Global best: 68747796901005.766, Runtime: 0.46663 seconds
2024/07/17 02:16:19 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 4, Current best: 68547715487770.52, Global best: 68547715487770.52, Runtime: 0.49274 seconds
2024/07/17 02:16:20 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 5, Current best: 68038346390200.945, Global best: 68038346390200.945, Runtime: 0.

Iteration 1: Best Fitness = 11576868394141.258
Iteration 2: Best Fitness = 11576868394141.258
Iteration 3: Best Fitness = 284103795466.7082
Iteration 4: Best Fitness = 18405025810.04015
Iteration 5: Best Fitness = 15571359759.595697
Iteration 6: Best Fitness = 15571359759.595697
Iteration 7: Best Fitness = 14571313824.180798
Iteration 8: Best Fitness = 13635958512.261768
Iteration 9: Best Fitness = 12247030556.366543
Iteration 10: Best Fitness = 12247030556.366543
Iteration 11: Best Fitness = 12089263318.403313
Iteration 12: Best Fitness = 12089263318.403313
Iteration 13: Best Fitness = 12089263318.403313
Iteration 14: Best Fitness = 12089263318.403313
Iteration 15: Best Fitness = 12089263318.403313
Iteration 16: Best Fitness = 12089263318.403313
Iteration 17: Best Fitness = 12089263318.403313
Iteration 18: Best Fitness = 12089263318.403313
Iteration 19: Best Fitness = 12089263318.403313
Iteration 20: Best Fitness = 12089263318.403313
Iteration 21: Best Fitness = 12089263318.403313
Ite

C:\Users\Lenovo\Desktop\EGWO\opfunu\cec_based\cec2017.py:1209: RuntimeWarning: invalid value encountered in divide
  ws = ws / np.sum(ws)


Iteration 90/100, Best Fitness: 3394797267420.31
Iteration 91/100, Best Fitness: 3394797267420.31
Iteration 92/100, Best Fitness: 3394797267420.31
Iteration 93/100, Best Fitness: 3394797267420.31
Iteration 94/100, Best Fitness: 3394797267420.31
Iteration 95/100, Best Fitness: 3391141267770.6616
Iteration 96/100, Best Fitness: 3391141267770.6616
Iteration 97/100, Best Fitness: 3391141267770.6616
Iteration 98/100, Best Fitness: 3391141267770.6616
Iteration 99/100, Best Fitness: 3385583029196.4585
Iteration 100/100, Best Fitness: 3381679530368.6973
*************APO****************
Best Position: [ -50.59324457   -6.73036316  -89.54832279   82.92778164   -4.88272026
   34.33243363   45.59002476  -43.73382576   51.93326917   80.24528602
   25.88160024  -32.58022091    3.00141036  -17.65663169   17.86700141
  -50.3764719    26.22175924   -7.82571283  -15.11584047   62.844569
 -102.31866944   -8.39949325   26.14770211   43.30745017  -17.22641553
  -44.04199314    4.57137164  -88.70695311   43

In [5]:
filename = "results.xlsx"
# 追加数据到Excel文件
append_to_excel(filename, F)

print(f"数据已成功写入 {filename}")

数据已成功写入 results.xlsx


In [6]:
print(min(H))

371585710.5891715


In [7]:
import numpy as np
from matplotlib import pyplot as plt

# In[]:
import opfunu  # 参考文档：https://github.com/thieu1995/opfunu
import mealpy
from APO import APO
from BSLO import BSLO
from MSGWO import MSGWO
from GWO import GWO
from HO import HO
from IVY import IVY
from BKA import BKA
from GA import GA
from NOA import NOA
from RBMO import RBMO
from SBOA import SBOA
# from mealpy.swarm_based import WOA, GWO
from mealpy import get_optimizer_by_name
from mealpy.evolutionary_based.GA import BaseGA
plt.rcParams['font.family'] = 'Times New Roman'

'''
适应度函数及维度dim的选择
cec函数名字格式：函数名+年份，比如要选择2022的F1函数，func_num = 'F1'+'2022'
cec2005：F1-F25, 可选 dim = 10, 30, 50
cec2008：F1-F7,  可选 2 <= dim <= 1000
cec2010：F1-F20, 可选 100 <= dim <= 1000
cec2013：F1-F28, 可选 dim = 2, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100
cec2014：F1-F30, 可选 dim = 10, 20, 30, 50, 100
cec2015：F1-F15, 可选 dim = 10, 30
cec2017：F1-F29, 可选 dim = 2, 10, 20, 30, 50, 100
cec2019：F1-F10, 可选 dim: F1=9,F2=16,F3=18,其他=10
cec2020：F1-F10, 可选 dim = 2, 5, 10, 15, 20, 30, 50, 100
cec2021：F1-F10, 可选 dim = 2, 10, 20
cec2022：F1-F12, 可选 dim = 2, 10, 20

'''
# fun_name = 'F22'  # 按需修改
year = '2017'  # 按需修改
func_num = fun_name + year
dim = 50  # 维度，根据cec函数 选择对应维度
epoch = 100  # 最大迭代次数
pop_size = 30  # 种群数量
'''定义的 cec函数 '''


def cec_fun(x):
    funcs = opfunu.get_functions_by_classname(func_num)
    func = funcs[0](ndim=dim)
    F = func.evaluate(x)
    return F


''' fit_func->目标函数, lb->下限, ub->上限 '''
problem_dict = {
    "fit_func": cec_fun,
    "lb": opfunu.get_functions_by_classname(func_num)[0](ndim=dim).lb.tolist(),
    "ub": opfunu.get_functions_by_classname(func_num)[0](ndim=dim).ub.tolist(),
    "minmax": "min",
}
import os
import pandas as pd
from openpyxl import load_workbook

def append_to_excel(filename, data_list):
    # 将列表转换为DataFrame
    df = pd.DataFrame([data_list])

    # 检查文件是否存在
    if not os.path.isfile(filename):
        # 如果文件不存在，创建新文件并写入数据，包含表头
        df.to_excel(filename, index=False, header=True)
    else:
        # 文件存在，尝试追加数据
        try:
            # 加载现有工作簿，并指定只读取数据，不读取样式等
            book = load_workbook(filename, data_only=True)

            # 确保Sheet1存在
            if 'Sheet1' not in book.sheetnames:
                book.create_sheet('Sheet1')

            # 找到Sheet1中的最后一行
            startrow = book['Sheet1'].max_row

            # 使用pandas的ExcelWriter以追加模式写入数据
            with pd.ExcelWriter(filename, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
                # 将数据追加到Sheet1
                df.to_excel(writer, sheet_name='Sheet1', startrow=startrow, index=False, header=False)

        except Exception as e:
            print(f"在追加数据时发生错误: {e}")

# # 示例数据列表
# data_list = [1, 2, 3, 4, 5]  # 注意这里应该是二维列表
# filename = "results.xlsx"
# # 追加数据到Excel文件
# append_to_excel(filename, data_list)
# 
# print(f"数据已成功写入 {filename}")

In [8]:
''' 调用优化算法 '''
''' 第二种方式，需：from mealpy import get_optimizer_by_name'''
woa_model = get_optimizer_by_name("OriginalWOA")(epoch, pop_size)
pso_model = get_optimizer_by_name("OriginalPSO")(epoch, pop_size)
ga_model = get_optimizer_by_name("BaseGA")(epoch, pop_size)

'''求解 cec函数 '''
H = []
F = []
f = []
for i in range(30):
    woa_best_x, woa_best_f = woa_model.solve(problem_dict)
    f.append(woa_best_f)
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    pso_best_x, pso_best_f = pso_model.solve(problem_dict)
    f.append(pso_best_f)
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve1, best_individual1 = GWO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve1[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve4, best_individual4 = HO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve4[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve5, best_individual5 = BSLO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve5[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve6, best_individual6 = RBMO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve6[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve7, best_individual7 = APO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve7[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve8, best_individual8 = IVY(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve8[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve9, best_individual9 = BKA(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve9[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve10, best_individual10 = GA(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve10[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    print("*****************************")
    fitness_curve, best_individual = MSGWO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
print(F)

2024/07/17 08:39:54 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: Solving single objective optimization problem.
2024/07/17 08:39:56 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 1, Current best: 5895159675823383.0, Global best: 5895159675823383.0, Runtime: 0.77293 seconds
2024/07/17 08:39:57 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 2, Current best: 5895159675823383.0, Global best: 5895159675823383.0, Runtime: 0.70219 seconds
2024/07/17 08:39:57 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 3, Current best: 4773985993279847.0, Global best: 4773985993279847.0, Runtime: 0.68237 seconds
2024/07/17 08:39:58 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 4, Current best: 2391939846849580.5, Global best: 2391939846849580.5, Runtime: 0.65889 seconds
2024/07/17 08:39:59 PM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 5, Current best: 1286609184762738.2, Global best: 1286609184762738.2, Runtime

Iteration 1: Best Fitness = 55023365910591.484
Iteration 2: Best Fitness = 55023365910591.484
Iteration 3: Best Fitness = 55023365910591.484
Iteration 4: Best Fitness = 19248014303418.016
Iteration 5: Best Fitness = 18714676967791.043
Iteration 6: Best Fitness = 13828498395384.355
Iteration 7: Best Fitness = 13828498395384.355
Iteration 8: Best Fitness = 13828498395384.355
Iteration 9: Best Fitness = 13828498395384.355
Iteration 10: Best Fitness = 13828498395384.355
Iteration 11: Best Fitness = 13828498395384.355
Iteration 12: Best Fitness = 13828498395384.355
Iteration 13: Best Fitness = 13828498395384.355
Iteration 14: Best Fitness = 13828498395384.355
Iteration 15: Best Fitness = 13828498395384.355
Iteration 16: Best Fitness = 13828498395384.355
Iteration 17: Best Fitness = 13828498395384.355
Iteration 18: Best Fitness = 13828498395384.355
Iteration 19: Best Fitness = 13828498395384.355
Iteration 20: Best Fitness = 13828498395384.355
Iteration 21: Best Fitness = 13828498395384.355
I

In [9]:
filename = "results.xlsx"
# 追加数据到Excel文件
append_to_excel(filename, F)

print(f"数据已成功写入 {filename}")

数据已成功写入 results.xlsx


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

# In[]:
import opfunu  # 参考文档：https://github.com/thieu1995/opfunu
import mealpy
from APO import APO
from BSLO import BSLO
from MSGWO import MSGWO
from GWO import GWO
from HO import HO
from IVY import IVY
from BKA import BKA
from GA import GA
from NOA import NOA
from RBMO import RBMO
from SBOA import SBOA
# from mealpy.swarm_based import WOA, GWO
from mealpy import get_optimizer_by_name
from mealpy.evolutionary_based.GA import BaseGA
plt.rcParams['font.family'] = 'Times New Roman'

'''
适应度函数及维度dim的选择
cec函数名字格式：函数名+年份，比如要选择2022的F1函数，func_num = 'F1'+'2022'
cec2005：F1-F25, 可选 dim = 10, 30, 50
cec2008：F1-F7,  可选 2 <= dim <= 1000
cec2010：F1-F20, 可选 100 <= dim <= 1000
cec2013：F1-F28, 可选 dim = 2, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100
cec2014：F1-F30, 可选 dim = 10, 20, 30, 50, 100
cec2015：F1-F15, 可选 dim = 10, 30
cec2017：F1-F29, 可选 dim = 2, 10, 20, 30, 50, 100
cec2019：F1-F10, 可选 dim: F1=9,F2=16,F3=18,其他=10
cec2020：F1-F10, 可选 dim = 2, 5, 10, 15, 20, 30, 50, 100
cec2021：F1-F10, 可选 dim = 2, 10, 20
cec2022：F1-F12, 可选 dim = 2, 10, 20

'''
# fun_name = 'F24'  # 按需修改
year = '2017'  # 按需修改
func_num = fun_name + year
dim = 100  # 维度，根据cec函数 选择对应维度
epoch = 100  # 最大迭代次数
pop_size = 30  # 种群数量
'''定义的 cec函数 '''


def cec_fun(x):
    funcs = opfunu.get_functions_by_classname(func_num)
    func = funcs[0](ndim=dim)
    F = func.evaluate(x)
    return F


''' fit_func->目标函数, lb->下限, ub->上限 '''
problem_dict = {
    "fit_func": cec_fun,
    "lb": opfunu.get_functions_by_classname(func_num)[0](ndim=dim).lb.tolist(),
    "ub": opfunu.get_functions_by_classname(func_num)[0](ndim=dim).ub.tolist(),
    "minmax": "min",
}
import os
import pandas as pd
from openpyxl import load_workbook

def append_to_excel(filename, data_list):
    # 将列表转换为DataFrame
    df = pd.DataFrame([data_list])

    # 检查文件是否存在
    if not os.path.isfile(filename):
        # 如果文件不存在，创建新文件并写入数据，包含表头
        df.to_excel(filename, index=False, header=True)
    else:
        # 文件存在，尝试追加数据
        try:
            # 加载现有工作簿，并指定只读取数据，不读取样式等
            book = load_workbook(filename, data_only=True)

            # 确保Sheet1存在
            if 'Sheet1' not in book.sheetnames:
                book.create_sheet('Sheet1')

            # 找到Sheet1中的最后一行
            startrow = book['Sheet1'].max_row

            # 使用pandas的ExcelWriter以追加模式写入数据
            with pd.ExcelWriter(filename, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
                # 将数据追加到Sheet1
                df.to_excel(writer, sheet_name='Sheet1', startrow=startrow, index=False, header=False)

        except Exception as e:
            print(f"在追加数据时发生错误: {e}")

# # 示例数据列表
# data_list = [1, 2, 3, 4, 5]  # 注意这里应该是二维列表
# filename = "results.xlsx"
# # 追加数据到Excel文件
# append_to_excel(filename, data_list)
# 
# print(f"数据已成功写入 {filename}")
''' 调用优化算法 '''
''' 第二种方式，需：from mealpy import get_optimizer_by_name'''
woa_model = get_optimizer_by_name("OriginalWOA")(epoch, pop_size)
pso_model = get_optimizer_by_name("OriginalPSO")(epoch, pop_size)
ga_model = get_optimizer_by_name("BaseGA")(epoch, pop_size)

'''求解 cec函数 '''
H = []
F = []
f = []
for i in range(30):
    woa_best_x, woa_best_f = woa_model.solve(problem_dict)
    f.append(woa_best_f)
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    pso_best_x, pso_best_f = pso_model.solve(problem_dict)
    f.append(pso_best_f)
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve1, best_individual1 = GWO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve1[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve4, best_individual4 = HO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve4[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve5, best_individual5 = BSLO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve5[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve6, best_individual6 = RBMO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve6[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve7, best_individual7 = APO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve7[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve8, best_individual8 = IVY(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve8[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve9, best_individual9 = BKA(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve9[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    fitness_curve10, best_individual10 = GA(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve10[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
f = []
for i in range(30):
    print("*****************************")
    fitness_curve, best_individual = MSGWO(cec_fun,dim,epoch,pop_size)
    f.append(fitness_curve[-1])
H.append(np.mean(f))
F.append(np.mean(f))
F.append(np.std(f))
print(F)

2024/07/18 09:16:24 AM, INFO, mealpy.swarm_based.WOA.OriginalWOA: Solving single objective optimization problem.
2024/07/18 09:16:29 AM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 1, Current best: 3.1256703642305536e+17, Global best: 3.1256703642305536e+17, Runtime: 2.72139 seconds
2024/07/18 09:16:32 AM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 2, Current best: 2.9779443394389184e+17, Global best: 2.9779443394389184e+17, Runtime: 2.76458 seconds
2024/07/18 09:16:34 AM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 3, Current best: 2.1128525933923443e+17, Global best: 2.1128525933923443e+17, Runtime: 2.67997 seconds
2024/07/18 09:16:37 AM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 4, Current best: 6.518588600444257e+16, Global best: 6.518588600444257e+16, Runtime: 2.61517 seconds
2024/07/18 09:16:42 AM, INFO, mealpy.swarm_based.WOA.OriginalWOA: >Problem: P, Epoch: 5, Current best: 6.518588600444257e+16, Global 

In [ ]:
filename = "results.xlsx"
# 追加数据到Excel文件
append_to_excel(filename, F)

print(f"数据已成功写入 {filename}")